In [5]:
import pandas as pd
import matplotlib.pyplot as plt
import re
from datetime import datetime
import numpy as np


# Path to your log file
log_file_list = ['../work/experiments/stellar-core/cp_4/tsm-sc-001/node2/stellar-core.log', \
                '../work/experiments/stellar-core/cp_8/tsm-sc-001/node2/stellar-core.log', \
                '../work/experiments/stellar-core/cp_16/tsm-sc-001/node2/stellar-core.log', ]


for log_file in log_file_list:
    print(log_file)
    # Regex patterns
    propose_pattern = re.compile(r"(\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}\.\d{3}).*Received PROPOSE block (\w+) at view (\d+)")
    commit_pattern  = re.compile(r"(\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}\.\d{3}).*Committed block (\w+) at view (\d+)")
    
    # Dictionaries to hold times
    propose_times = {}
    commit_times = {}
    
    # Parse log file
    with open(log_file) as f:
        for line in f:
            pmatch = propose_pattern.search(line)
            if pmatch:
                ts = datetime.strptime(pmatch.group(1), "%Y-%m-%dT%H:%M:%S.%f")
                block = pmatch.group(2)
                view = int(pmatch.group(3))
                propose_times[(block, view)] = ts
            
            cmatch = commit_pattern.search(line)
            if cmatch:
                ts = datetime.strptime(cmatch.group(1), "%Y-%m-%dT%H:%M:%S.%f")
                block = cmatch.group(2)
                view = int(cmatch.group(3))
                commit_times[(block, view)] = ts
    
    # Match proposals to commits → compute latency
    latencies = []
    commit_timestamps = []
    for key, ctime in commit_times.items():
        if key in propose_times:
            latency = (ctime - propose_times[key]).total_seconds()
            latencies.append({
                "block": key[0],
                "view": key[1],
                "commit_time": ctime,
                "latency": latency
            })
            commit_timestamps.append(ctime)
    
    if not commit_timestamps:
        print("No commits found.")
        exit()
    
    # Normalize times to seconds since start
    start_time = min(commit_timestamps + list(propose_times.values()))
    commit_seconds = [(t - start_time).total_seconds() for t in commit_timestamps]
    
    # DataFrame for throughput
    df = pd.DataFrame({"time_sec": commit_seconds, "count": 1})
    df.set_index("time_sec", inplace=True)
    
    # Throughput = commits per second (resampled)
    throughput = df["count"].groupby(df.index.round()).sum()
    
    # DataFrame for latencies
    lat_df = pd.DataFrame(latencies)
    lat_df["time_sec"] = lat_df["commit_time"].apply(lambda t: (t - start_time).total_seconds())
    
    # # ---- Plot Throughput ----
    # plt.figure(figsize=(10, 4))
    # plt.plot(throughput.index, throughput.values, "-", label="Throughput (blocks/sec)")
    
    # plt.xlabel("Time (s)")
    # plt.ylabel("Blocks/sec")
    # plt.title("Throughput vs Time")
    # # plt.grid(True)
    # plt.legend()
    # plt.tight_layout()
    # plt.savefig("throughput.png", dpi=150)
    # plt.show()
    
    # # ---- Plot Latency ----
    # plt.figure(figsize=(10, 4))
    # plt.plot(lat_df["time_sec"], lat_df["latency"], "-", color="r", label="Latency (s)")
    # plt.xlabel("Time (s)")
    # plt.ylabel("Latency (s)")
    # plt.title("Proposal → Commit Latency")
    # # plt.grid(True)
    # plt.legend()
    # plt.tight_layout()
    # plt.savefig("latency.png", dpi=150)
    # plt.show()
    print(np.average(throughput.values[-60:]))
    print(np.average(np.array(lat_df["latency"])[-60:]))

../work/experiments/stellar-core/cp_4/tsm-sc-001/node2/stellar-core.log
75.46666666666667
0.0017666666666666666
../work/experiments/stellar-core/cp_8/tsm-sc-001/node2/stellar-core.log
69.96666666666667
0.0028500000000000005
../work/experiments/stellar-core/cp_16/tsm-sc-001/node2/stellar-core.log
No commits found.


ValueError: min() iterable argument is empty

In [3]:

len(throughput.values)

337